In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam, RMSprop, Nadam
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import pandas as pd
import uuid
import random
import os
import gc

In [2]:
SEED = 69
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

train_df = pd.read_csv('california_housing_train.csv')
test_df = pd.read_csv('california_housing_test.csv')

features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
            'total_bedrooms', 'population', 'households', 'median_income']
target = 'median_house_value'

x_train_full = train_df[features].values
y_train_full = train_df[target].values
x_test = test_df[features].values
y_test = test_df[target].values


mean = x_train_full.mean(axis=0)
std = x_train_full.std(axis=0)
X_train = (x_train_full - mean) / std
X_test = (x_test - mean) / std

y_train = y_train_full

In [ ]:
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)


param_grid = {
    'n_layers': [2, 3, 4, 5],
    'units': [128, 256, 512, 1024],
    'activation': ['relu', 'tanh', 'elu'],
    'initializer': ['he_normal', 'he_uniform', 'glorot_uniform', 'glorot_normal'],
    'dropout_rate': [0.0, 0.1, 0.2, 0.3, 0.4],
    'l1_reg': [0.0, 1e-5, 1e-4, 1e-3],
    'l2_reg': [0.0, 1e-5, 1e-4, 1e-3],
    'batch_norm': [True, False],
    'optimizer': ['adam', 'rmsprop', 'nadam'],
    'learning_rate': [1e-2, 1e-3, 1e-4],
    'batch_size': [64, 128, 256]
}

def clear_memory():
    """Очистка памяти TensorFlow и сбор мусора"""
    tf.keras.backend.clear_session()
    gc.collect()

def get_compatible_initializers(activation):
    """Возвращает совместимые инициализаторы для данной функции активации"""
    if activation == 'relu':
        return ['he_normal', 'he_uniform']
    elif activation == 'elu':
        return ['he_normal', 'he_uniform']
    elif activation == 'tanh':
        return ['glorot_uniform', 'glorot_normal']
    else:
        return ['he_normal', 'he_uniform', 'glorot_uniform', 'glorot_normal']

def generate_architecture(n_layers, max_units=2048):
    """Генерирует архитектуру с постепенным уменьшением нейронов"""
    units = []
    current_units = max_units

    for i in range(n_layers):
        # Выбираем количество нейронов для текущего слоя не больше предыдущего
        available_units = [u for u in param_grid['units'] if u <= current_units]
        if not available_units:
            available_units = [current_units]

        layer_units = random.choice(available_units)
        units.append(layer_units)
        current_units = layer_units

    return units

def create_model(params, units_list):
    model = models.Sequential()
    model.add(layers.Input(shape=(X_train.shape[1],)))

    for i, units in enumerate(units_list):
        kernel_regularizer = regularizers.l1_l2(
            l1=params['l1_reg'],
            l2=params['l2_reg']
        )

        model.add(layers.Dense(
            units,
            activation=None, 
            kernel_initializer=params['initializer'],
            kernel_regularizer=kernel_regularizer
        ))

        model.add(layers.Activation(params['activation']))

        if params['batch_norm']:
            model.add(layers.BatchNormalization())

        if params['dropout_rate'] > 0:
            model.add(layers.Dropout(params['dropout_rate']))

    model.add(layers.Dense(1))
    return model

def get_optimizer(params):
    if params['optimizer'] == 'adam':
        return Adam(learning_rate=params['learning_rate'])
    elif params['optimizer'] == 'rmsprop':
        return RMSprop(learning_rate=params['learning_rate'])
    elif params['optimizer'] == 'nadam':
        return Nadam(learning_rate=params['learning_rate'])

def save_model_and_history(model, history, experiment_id):
    """Сохраняет модель и историю обучения"""
    model_path = f'models/model_{experiment_id}.keras'
    history_path = f'models/history_{experiment_id}.pkl'

    model.save(model_path)

    import pickle
    with open(history_path, 'wb') as f:
        pickle.dump(history.history, f)

    return model_path, history_path

def generate_random_params(n_experiments=500):
    experiments = []

    for _ in range(n_experiments):
        n_layers = random.choice(param_grid['n_layers'])
        activation = random.choice(param_grid['activation'])

        # Выбираем совместимый инициализатор
        compatible_initializers = get_compatible_initializers(activation)
        initializer = random.choice(compatible_initializers)

        units_list = generate_architecture(n_layers)

        params = {
            'n_layers': n_layers,
            'units_list': units_list,
            'activation': activation,
            'initializer': initializer,
            'dropout_rate': random.choice(param_grid['dropout_rate']),
            'l1_reg': random.choice(param_grid['l1_reg']),
            'l2_reg': random.choice(param_grid['l2_reg']),
            'batch_norm': random.choice(param_grid['batch_norm']),
            'optimizer': random.choice(param_grid['optimizer']),
            'learning_rate': random.choice(param_grid['learning_rate']),
            'batch_size': random.choice(param_grid['batch_size'])
        }
        experiments.append(params)

    return experiments

# Генерируем
all_experiments = generate_random_params(500)

results = []
target_mae = 30000

print(f"Starting {len(all_experiments)} experiments...")
print(f"Models will be saved in 'models' folder")

for i, params in enumerate(all_experiments):
    print(f"Training model {i+1}/{len(all_experiments)}")
    print(f"Architecture: {params['units_list']}, Activation: {params['activation']}")

    try:
        # Очистка памяти перед созданием новой модели
        clear_memory()

        model = create_model(params, params['units_list'])
        optimizer = get_optimizer(params)

        model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=50,
            restore_best_weights=True,
            min_delta=1
        )

        history = model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=400,
            batch_size=params['batch_size'],
            callbacks=[early_stop],
            verbose=0
        )

        val_mae = history.history['val_mae'][-1]
        train_loss, train_mae = model.evaluate(X_train, y_train, verbose=0)
        test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

        # Генерация ID эксперимента
        experiment_id = str(uuid.uuid4())[:8]

        model_path, history_path = save_model_and_history(model, history, experiment_id)

        result = {
            'experiment_id': experiment_id,
            'n_layers': params['n_layers'],
            'units_list': str(params['units_list']),
            'activation': params['activation'],
            'initializer': params['initializer'],
            'dropout_rate': params['dropout_rate'],
            'l1_reg': params['l1_reg'],
            'l2_reg': params['l2_reg'],
            'batch_norm': params['batch_norm'],
            'optimizer': params['optimizer'],
            'learning_rate': params['learning_rate'],
            'batch_size': params['batch_size'],
            'train_mae': float(train_mae),
            'test_mae': float(test_mae),
            'val_mae': float(val_mae),
            'epochs_trained': len(history.history['loss']),
            'final_learning_rate': float(history.history.get('lr', [params['learning_rate']])[-1]),
            'model_path': model_path,
            'history_path': history_path
        }
        results.append(result)

        print(f"  Train MAE: {train_mae:.4f}, Test MAE: {test_mae:.4f}")
        print(f"  Model saved: {model_path}")

        # Явная очистка памяти после каждого эксперимента
        del model
        clear_memory()

        if test_mae <= target_mae:
            print(f" Target MAE achieved at experiment {i+1}")
            break

    except Exception as e:
        print(f" Error in experiment {i+1}: {str(e)}")
        clear_memory()
        continue

    # Промежуточное
    if (i + 1) % 10 == 0:
        df = pd.DataFrame(results)
        df.to_csv(f'results/nn_experiments_interim_{i+1}.csv', index=False)
        print(f"Interim results saved at experiment {i+1}")

df = pd.DataFrame(results)
df.to_csv('results/nn_experiments_final.csv', index=False)

if results:
    top_models = sorted(results, key=lambda x: x['test_mae'])[:5]
    top_models_df = pd.DataFrame(top_models)
    top_models_df.to_csv('results/top_5_models.csv', index=False)

print("\n=== RESULTS SUMMARY ===")
if results:
    best_result = min(results, key=lambda x: x['test_mae'])
    print(f"Best Test MAE: {best_result['test_mae']:.4f}")
    print(f"Best Architecture: {best_result['units_list']}")
    print(f"Best Parameters: Activation={best_result['activation']}, "
          f"Optimizer={best_result['optimizer']}, LR={best_result['learning_rate']}")
    print(f"Best Model: {best_result['model_path']}")

    test_maes = [r['test_mae'] for r in results]
    print(f"Average Test MAE: {np.mean(test_maes):.4f} ± {np.std(test_maes):.4f}")
    print(f"Experiments completed: {len(results)}")

print("All experiments completed!")
print(f" Models saved in: models/")
print(f"Results saved in: results/")

Starting 500 experiments...
Models will be saved in 'models' folder
Training model 1/500
Architecture: [128, 128], Activation: relu
  Train MAE: 45077.0859, Test MAE: 45240.0391
  Model saved: models/model_e29df776.keras
Training model 2/500
Architecture: [256, 256, 256, 128], Activation: elu
  Train MAE: 35187.2617, Test MAE: 37450.7109
  Model saved: models/model_a586f58b.keras
Training model 3/500
Architecture: [1024, 128, 128], Activation: relu
  Train MAE: 37387.7031, Test MAE: 38417.5234
  Model saved: models/model_2133167b.keras
Training model 4/500
Architecture: [512, 128, 128], Activation: relu
  Train MAE: 44460.4375, Test MAE: 44726.1367
  Model saved: models/model_142c48b1.keras
Training model 5/500
Architecture: [512, 128], Activation: tanh
  Train MAE: 179650.5156, Test MAE: 178189.8750
  Model saved: models/model_7684815e.keras
Training model 6/500
Architecture: [1024, 1024, 1024, 512, 512], Activation: relu
  Train MAE: 39421.7656, Test MAE: 40971.0586
  Model saved: mo

KeyboardInterrupt: 

In [ ]:
path_to_model = "D:\\Downloads\\summary_neuron3\\models\\model_2c31bfdc.keras"
model = tf.keras.models.load_model(path_to_model)

test_loss, test_mae = model.evaluate(X_test, y_test, verbose=1)
print(f"\nTest MAE: {test_mae:.2f}")

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3006139392.0000 - mae: 35425.0273
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3006139392.0000 - mae: 35425.0273

Test MAE: 35425.03

Test MAE: 35425.03
